In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    median_absolute_error,
    r2_score,
    explained_variance_score
)

# ---------------------------
# 1. Load raw CSVs
# ---------------------------

path_apple = '/Users/ambervo/Library/CloudStorage/OneDrive-Personal/Documents/GMBA 3/FinTech/Group Project/data/Apple Stock Price History.csv'
path_nvda  = '/Users/ambervo/Library/CloudStorage/OneDrive-Personal/Documents/GMBA 3/FinTech/Group Project/data/NVIDIA Stock Price History.csv'
path_tsla  = '/Users/ambervo/Library/CloudStorage/OneDrive-Personal/Documents/GMBA 3/FinTech/Group Project/data/Tesla Stock Price History.csv'
path_spx   = '/Users/ambervo/Library/CloudStorage/OneDrive-Personal/Documents/GMBA 3/FinTech/Group Project/data/S&P 500 - Historical Data.csv'

apple_raw = pd.read_csv(path_apple)
nvda_raw  = pd.read_csv(path_nvda)
tsla_raw  = pd.read_csv(path_tsla)
spx_raw   = pd.read_csv(path_spx)

def detect_close_column(df):
    """Try to guess the close/price column name."""
    candidates = ["Close", "Adj Close", "Adj Close*", "Price", "Last"]
    for c in candidates:
        if c in df.columns:
            return c
    return df.columns[-1]

def prep_asset(df, name):
    """
    Prepare one asset:
    - parse Date
    - detect price column
    - compute log return
    - compute RV20, RV60, RV120 (annualized)
    """
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

    close_col = detect_close_column(df)
    df[close_col] = (
        df[close_col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

    # log return
    df[f"{name}_logret"] = np.log(df[close_col]).diff()

    # realized volatility (annualized)
    for win in (20, 60, 120):
        df[f"{name}_RV{win}"] = np.sqrt(252) * df[f"{name}_logret"].rolling(win).std()

    keep_cols = ["Date"] + [c for c in df.columns if c.startswith(name)]
    return df[keep_cols]

AAPL = prep_asset(apple_raw, "AAPL")
NVDA = prep_asset(nvda_raw,  "NVDA")
TSLA = prep_asset(tsla_raw,  "TSLA")
SPX  = prep_asset(spx_raw,   "SPX")

# ---------------------------
# 2. Merge all assets
# ---------------------------

data = (
    SPX.merge(AAPL, on="Date")
       .merge(NVDA, on="Date")
       .merge(TSLA, on="Date")
       .sort_values("Date")
)

# Target: SPX_RV60
data["y"] = data["SPX_RV60"]

# All candidate columns except Date and y
all_candidate_cols = [c for c in data.columns if c not in ["Date", "y"]]

# Remove SPX_RV60 from features to get 15 predictors
feature_cols = [c for c in all_candidate_cols if c != "SPX_RV60"]

print("All candidate columns:", all_candidate_cols)
print("Final feature columns (should be 15):", feature_cols)
print("Number of features:", len(feature_cols))

# Lag features by 1 day to avoid look-ahead bias
for c in feature_cols:
    data[c] = data[c].shift(1)

# Drop NaNs from diff/rolling/shift
data = data.dropna().reset_index(drop=True)

X_df = data[feature_cols]
y = data["y"].values

print("Number of observations:", len(y))


All candidate columns: ['SPX_logret', 'SPX_RV20', 'SPX_RV60', 'SPX_RV120', 'AAPL_logret', 'AAPL_RV20', 'AAPL_RV60', 'AAPL_RV120', 'NVDA_logret', 'NVDA_RV20', 'NVDA_RV60', 'NVDA_RV120', 'TSLA_logret', 'TSLA_RV20', 'TSLA_RV60', 'TSLA_RV120']
Final feature columns (should be 15): ['SPX_logret', 'SPX_RV20', 'SPX_RV120', 'AAPL_logret', 'AAPL_RV20', 'AAPL_RV60', 'AAPL_RV120', 'NVDA_logret', 'NVDA_RV20', 'NVDA_RV60', 'NVDA_RV120', 'TSLA_logret', 'TSLA_RV20', 'TSLA_RV60', 'TSLA_RV120']
Number of features: 15
Number of observations: 3530


In [2]:
# ---------------------------
# 3. TimeSeriesSplit
# ---------------------------

tscv = TimeSeriesSplit(n_splits=5)

# ---------------------------
# 4. Metrics function
# ---------------------------

def compute_metrics(y_true, y_pred):
    """Compute all metrics for one fold."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    evs  = explained_variance_score(y_true, y_pred)

    corr = np.corrcoef(y_true, y_pred)[0, 1]

    # QLIKE (ensure positivity)
    eps = 1e-8
    y_true_pos = np.clip(y_true, eps, None)
    y_pred_pos = np.clip(y_pred, eps, None)
    ratio = y_true_pos / y_pred_pos
    qlike = np.mean(ratio - np.log(ratio) - 1)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "MedAE": medae,
        "R2": r2,
        "ExplVar": evs,
        "Corr": corr,
        "QLIKE": qlike
    }

# ---------------------------
# 5. Benchmark model: Linear Regression with all features
# ---------------------------

from collections import defaultdict

benchmark_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

X_all = X_df.values

fold_metrics = defaultdict(list)

for train_idx, test_idx in tscv.split(X_all):
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    benchmark_model.fit(X_train, y_train)
    y_pred = benchmark_model.predict(X_test)

    m = compute_metrics(y_test, y_pred)
    for key, value in m.items():
        fold_metrics[key].append(value)

# Aggregate across folds
benchmark_summary = {
    "FeatureSelection": "AllFS",       # all 15 features
    "Model": "LinearRegression",
    "n_features": len(feature_cols)
}
for key, vals in fold_metrics.items():
    benchmark_summary[f"{key}_mean"] = np.mean(vals)
    benchmark_summary[f"{key}_std"]  = np.std(vals)

benchmark_results_df = pd.DataFrame([benchmark_summary])

# nice ordering
cols_order = [
    "FeatureSelection", "Model", "n_features",
    "RMSE_mean", "RMSE_std",
    "MAE_mean", "MAE_std",
    "MAPE_mean", "MAPE_std",
    "MedAE_mean", "MedAE_std",
    "R2_mean", "R2_std",
    "ExplVar_mean", "ExplVar_std",
    "Corr_mean", "Corr_std",
    "QLIKE_mean", "QLIKE_std"
]

benchmark_results_df = benchmark_results_df[cols_order]

# Show and save
print(benchmark_results_df.round(4))
benchmark_results_df.round(6).to_csv("benchmark_results.csv", index=False)


  FeatureSelection             Model  n_features  RMSE_mean  RMSE_std  \
0            AllFS  LinearRegression          15     0.0267    0.0101   

   MAE_mean  MAE_std  MAPE_mean  MAPE_std  MedAE_mean  MedAE_std  R2_mean  \
0    0.0212   0.0091     0.1655    0.0879      0.0177     0.0095  -0.2554   

   R2_std  ExplVar_mean  ExplVar_std  Corr_mean  Corr_std  QLIKE_mean  \
0  2.3145       -0.2374       2.2806      0.841    0.2225      0.0423   

   QLIKE_std  
0     0.0587  
